# Graph Sanity Check
Validates the Sahel GNN adjacency matrix before handing off to the model.

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from pathlib import Path
from geopy.distance import geodesic

DATA_DIR = Path("../data")


#  Load data 

In [ ]:
nodes      = pd.read_csv(DATA_DIR / "nodes.csv")
A_norm     = np.load(DATA_DIR / "adjacency_matrix.npy")
edge_index = np.load(DATA_DIR / "edge_index.npy")
edge_weight= np.load(DATA_DIR / "edge_weight.npy")

print(f"Nodes:       {len(nodes)}")
print(f"A_norm:      {A_norm.shape}")
print(f"Edge index:  {edge_index.shape}  ({edge_index.shape[1]} directed edges)")
print(f"Edge weight: {edge_weight.shape}")

## 1. Shape & value checks

In [ ]:
N = len(nodes)
assert A_norm.shape == (N, N), f"Shape mismatch: {A_norm.shape}"
assert A_norm.min() >= 0,      f"Negative weight: {A_norm.min()}"
assert not np.any(np.isnan(A_norm)), "NaN in A"
print(f"Shape: {A_norm.shape}")
print(f"Weight range: [{A_norm.min():.4f}, {A_norm.max():.4f}]")
print(f"No NaN / Inf")

## 2. Build NetworkX graph for analysis

In [ ]:
# Rebuild raw (un-normalized) graph from edge_index for inspection
G = nx.Graph()
for i, row in nodes.iterrows():
    G.add_node(i, name=row["name"], lat=row["lat"], lon=row["lon"], country=row["country"])

# Use only upper triangle to avoid duplicate edges
seen = set()
for k in range(edge_index.shape[1]):
    u, v = int(edge_index[0, k]), int(edge_index[1, k])
    if u != v and (min(u,v), max(u,v)) not in seen:
        seen.add((min(u,v), max(u,v)))
        G.add_edge(u, v, weight=float(edge_weight[k]))

print(f"NetworkX graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
print(f"Connected: {nx.is_connected(G)}")

## 3. Key corridor checks

In [ ]:
node_idx = {row["id"]: i for i, row in nodes.iterrows()}

CORRIDORS = [
    ("mopti",       "gao",          "Mali central corridor"),
    ("gao",         "kidal",        "Mali northeast corridor"),
    ("agadez",      "diffa",        "Niger corridor"),
    ("ouagadougou", "dori",         "Burkina corridor"),
    ("niamey",      "agadez",       "Niger north-south"),
    ("mopti",       "ouagadougou",  "Mali-Burkina link"),
]

for src, dst, label in CORRIDORS:
    if src in node_idx and dst in node_idx:
        i, j = node_idx[src], node_idx[dst]
        w = A_norm[i, j]
        # Also compute raw distance
        dist = geodesic(
            (nodes.iloc[i]["lat"], nodes.iloc[i]["lon"]),
            (nodes.iloc[j]["lat"], nodes.iloc[j]["lon"])
        ).km
        status = "PASSED" if w > 0 else "FAILED"
        print(f"  {status} {src:15s} ↔ {dst:15s}  dist={dist:.0f}km  weight={w:.4f}  ({label})")

## 4. Degree distribution

In [ ]:
degrees = dict(G.degree())
deg_vals = list(degrees.values())

print(f"\nDegree statistics:")
print(f"  Mean:   {np.mean(deg_vals):.1f}")
print(f"  Median: {np.median(deg_vals):.0f}")
print(f"  Min:    {min(deg_vals)} ({nodes.iloc[np.argmin(deg_vals)]['name']})")
print(f"  Max:    {max(deg_vals)} ({nodes.iloc[np.argmax(deg_vals)]['name']})")

# Top 5 most connected nodes
top5 = sorted(degrees.items(), key=lambda x: x[1], reverse=True)[:5]
print(f"\nTop 5 most connected nodes:")
for idx, deg in top5:
    print(f"  {nodes.iloc[idx]['name']:20s}  degree={deg}")

## 5. Graph visualization (geographic layout)

In [ ]:
pos    = {i: (row["lon"], row["lat"]) for i, row in nodes.iterrows()}
labels = {i: row["name"] for i, row in nodes.iterrows()}

country_colors = {
    "Mali":         "#E63946",
    "Niger":        "#F4A261",
    "Burkina Faso": "#2A9D8F",
    "Chad":         "#457B9D",
    "Mauritania":   "#6A4C93",
}
node_colors = [country_colors.get(nodes.iloc[i]["country"], "#888") for i in G.nodes()]
edge_weights_list = [G[u][v]["weight"] for u, v in G.edges()]

fig, axes = plt.subplots(1, 2, figsize=(18, 8))
fig.patch.set_facecolor("#1a1a2e")

# Left: geographic layout
ax = axes[0]
ax.set_facecolor("#1a1a2e")
nx.draw_networkx_edges(G, pos, ax=ax, width=[1 + w*3 for w in edge_weights_list],
                       edge_color="white", alpha=0.3)
nx.draw_networkx_nodes(G, pos, ax=ax, node_color=node_colors, node_size=150,
                       edgecolors="white", linewidths=1)
nx.draw_networkx_labels(G, pos, labels, ax=ax, font_size=6, font_color="white")
ax.set_title("Geographic Layout", color="white", fontsize=11)
ax.tick_params(colors="#aaa")

# Right: degree distribution
ax2 = axes[1]
ax2.set_facecolor("#1a1a2e")
ax2.hist(deg_vals, bins=range(min(deg_vals), max(deg_vals)+2),
         color="#E63946", edgecolor="white", linewidth=0.5, alpha=0.85)
ax2.set_xlabel("Node Degree", color="white")
ax2.set_ylabel("Count", color="white")
ax2.set_title("Degree Distribution", color="white", fontsize=11)
ax2.tick_params(colors="white")

plt.suptitle("Sahel GNN — Graph Sanity Check", color="white", fontsize=13)
plt.tight_layout()
plt.savefig("graph_sanity_check_output.png", dpi=150, bbox_inches="tight",
            facecolor="#1a1a2e")
plt.show()
print("Visualization saved")


## 6. Final verdict

In [ ]:

checks = {
    "Shape (25×25)":      A_norm.shape == (N, N),
    "No NaN":             not np.any(np.isnan(A_norm)),
    "Non-negative":       A_norm.min() >= 0,
    "Connected graph":    nx.is_connected(G),
    "Mopti-Gao edge":     A_norm[node_idx.get("mopti", 0), node_idx.get("gao", 0)] > 0,
    "All nodes present":  G.number_of_nodes() == N,
}

print("\n── Final sanity check results ──────────────────────────")
all_pass = True
for check, result in checks.items():
    status = "PASSED" if result else "FAILED"
    print(f"  {status}  {check}")
    if not result:
        all_pass = False

print("────────────────────────────────────────────────────────")
if all_pass:
    print("ALL CHECKS PASSED — graph is ready for the model")
else:
    print("SOME CHECKS FAILED — review output above")
